# ChillProp Usage Guide

This notebook introduces the public API, the internal modeling conventions, and the current compatibility envelope for contributors and advanced users.

## Setup

The notebook assumes a local development install or an installed package from PyPI. The examples use JAX arrays where vectorization or differentiation is relevant.

In [ ]:
import jax
import jax.numpy as jnp
import chillprop.highlevel as CH

## Basic `PropsSI` queries

`PropsSI` accepts either a two-argument trivial-property form or a six-argument state-property form. Internal state solves use molar density, while the high-level API exposes the same mass-basis aliases commonly used in CoolProp.

In [ ]:
rho = CH.PropsSI("D", "T", 300.0, "P", 1e6, "Air")
h = CH.PropsSI("H", "T", 300.0, "P", 1e6, "Nitrogen")
Tcrit = CH.PropsSI("Tcrit", "Propane")

print("Air density [kg/m^3]:", rho)
print("Nitrogen enthalpy [J/kg]:", h)
print("Propane critical temperature [K]:", Tcrit)

## Batched evaluation

Vectorized inputs are broadcast and evaluated through the same compatibility layer. This is the recommended entry point for parameter sweeps and JAX-native batch workflows.

In [ ]:
T = jnp.array([280.0, 300.0, 320.0])
P = jnp.array([5e5, 1e6, 2e6])

rho = CH.PropsSI("D", "T", T, "P", P, "Nitrogen")
mu = CH.PropsSI("V", "T", T, "P", P, "Nitrogen")

print("Density [kg/m^3]:", rho)
print("Viscosity [Pa*s]:", mu)

## Differentiation through property calls

The thermodynamic kernels and the `PT` density solve are differentiable. This makes ChillProp suitable for gradient-based workflows as long as the state specification stays inside the implemented compatibility envelope.

In [ ]:
def enthalpy_at_pressure(p):
    return CH.PropsSI("H", "T", 400.0, "P", p, "Nitrogen")

dh_dp = jax.grad(enthalpy_at_pressure)(5e6)
print("dH/dP at constant T [J/kg/Pa]:", dh_dp)

## `AbstractState` workflow

`AbstractState` provides a stateful interface for supported input pairs and keyed outputs. The current implementation intentionally covers a subset of CoolProp rather than the entire `AbstractState` surface.

In [ ]:
state = CH.AbstractState("HEOS", "Nitrogen")
state.update(CH.PT_INPUTS, 1e6, 300.0)

print("rhomolar [mol/m^3]:", state.rhomolar())
print("hmolar [J/mol]:", state.hmolar())
print("viscosity [Pa*s]:", state.viscosity())
print("conductivity [W/m/K]:", state.conductivity())

## Two-phase queries

For supported pure fluids, `TQ` state specification uses the saturation solve in `phases.py`. High-level outputs such as `H`, `S`, `U`, `P`, and `Q` are blended using the saturated liquid and vapor states.

In [ ]:
q = 0.5
T_sat = 110.0

rho_molar = CH.PropsSI("Dmolar", "T", T_sat, "Q", q, "Nitrogen")
h_molar = CH.PropsSI("Hmolar", "T", T_sat, "Q", q, "Nitrogen")

print("Saturated mixture density [mol/m^3]:", rho_molar)
print("Saturated mixture enthalpy [J/mol]:", h_molar)

## Architecture map

The public API is a thin adapter over a set of functional modules:

- `parameters.py`: JSON parsing and typed fluid metadata
- `heos.py`: ideal and residual Helmholtz terms
- `core.py`: thermodynamic properties from Helmholtz derivatives
- `phases.py`: ancillaries, saturation, and phase classification
- `solver.py`: differentiable state inversion
- `transport.py`: viscosity and conductivity models
- `highlevel.py`: CoolProp-style API translation, caching, and dispatch

A typical `PropsSI` call resolves a fluid, solves for `(rho, T)`, then dispatches to a property evaluator.

## Conventions for contributors

- Internal density is molar density in `mol/m^3`
- Energy-like low-level properties are molar unless a mass-basis alias is requested at the API boundary
- Reduced variables follow `tau = Tr / T` and `delta = rho / rhor`
- New public outputs generally require changes in `_INPUT_ALIASES`, `_solve_state()`, `_evaluate_output()`, tests, and documentation

## Current compatibility limits

The package does not currently implement the full CoolProp feature surface. The most important limits are:

- only the default backend and `HEOS` are supported
- mixtures are not implemented
- derivative-string outputs are not implemented
- phase-imposed input keys are not implemented
- only a subset of `AbstractState.update()` input pairs is supported

The maintained gap list lives in `docs/wiki/Implementation_Gaps.md`.

## Suggested next reading

- `docs/wiki/Architecture.md` for module-level design details
- `docs/wiki/Validation.md` for parity statistics and plots
- `tests/test_highlevel_pure_jax.py` for the current public compatibility contract